# DPCNN with GloVe

Train the convolutional model with GloVe embeddings and square-root class weights.

Run from the repository root.

# DPCNN + GloVe — Square-Root Weighting
No terminal is required. Run 00_prepare_data.ipynb first, then run this notebook from top to bottom.
The first cell contains parameters, the second contains the training implementation and usually needs no changes, and the third starts training.
Training fits the embeddings and model, validation selects the epoch, and test evaluation is enabled only for the final independent test.
The two models with the same embedding share fixed vectors; all six combinations must use the same inputs, processing, and comparison rules.


In [ ]:
# Independent model program: no imports from the old experiment workspace.

import argparse, hashlib, json, random, time
from pathlib import Path
from collections import Counter
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import classification_report, confusion_matrix
import seaborn  as sns
import matplotlib.pyplot as plt

MODEL_NAME = 'DPCNN'
EMBEDDING_METHOD = 'glove'
# ===== Parameter Area: Usually the Only Section to Modify =====
SEED = 42
DIMENSION = 48          # Embedding dimension: keep identical across all six combinations
MAX_LENGTH = 96         # Maximum words per text: keep identical across all six combinations
WINDOW = 4             # Embedding context window
MAX_VOCAB = 30000      # Vocabulary size

EPOCHS = 5           # Maximum training epochs; select the epoch by validation Macro-F1
EMBEDDING_EPOCHS = 5    # Number of embedding training epochs
BATCH_SIZE = 128     # Batch size
LEARNING_RATE = 0.0005     # Learning rate
WEIGHT_DECAY = 0.01       # Weight decay
GRADIENT_CLIP = 5         # Gradient norm clipping
MIN_COUNT = 2          # Minimum word frequency for the vocabulary
CHANNELS = 128          # DPCNN convolution channels / TextRCNN projection dimension
BLOCKS = 4             # DPCNN only: number of residual blocks
KERNEL_SIZE = 3        # DPCNN only: positive odd-sized convolution kernel
WEIGHT_ALPHA = 0.5  # 0=unweighted; 0.5=square-root weighting
# ===== End of Parameter Area =====
NAMES = ['low', 'moderate', 'good', 'excellent']
PACKAGE_DIR = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()




Utility Functions

In [ ]:
"""Map the original 1-10 rating to a four-class label."""
def label_of(rating):
    assert 1 <= rating <= 10
    return 0 if rating <= 3 else 1 if rating <= 6 else 2 if rating <= 8 else 3

"""Convert text word lists into token-index tensors and build the dataset."""
def encode(rows, vocabulary, unknown_id):
    tokens = torch.zeros((len(rows), MAX_LENGTH), dtype=torch.long)
    labels = torch.tensor([label for _, label in rows], dtype=torch.long)
    for index, (review, _) in enumerate(rows):
        selected = review[:MAX_LENGTH]
        tokens[index, :len(selected)] = torch.tensor([vocabulary.get(word, unknown_id) for word in selected])
    return TensorDataset(tokens, labels)

"""Evaluate the model: calculate accuracy, precision, F1, recall, and other metrics."""
def evaluate(model,embedding,loader):
    model.eval()
    probability=[]; truth=[]
    with torch.no_grad():
        for ids,y in loader:
            probability.append(model(embedding(ids),ids.ne(0)).softmax(1).numpy())
            truth.extend(y.tolist())
    target=np.asarray(truth); probability=np.concatenate(probability)
    predicted=probability.argmax(1)
    report=classification_report(target,predicted,labels=range(4),target_names=NAMES,output_dict=True,zero_division=0)
    metrics=dict(accuracy=float((target==predicted).mean()),macro_f1=report['macro avg']['f1-score'],
                 precision_macro=report['macro avg']['precision'],recall_macro=report['macro avg']['recall'],
                 classification_report=report,confusion_matrix=confusion_matrix(target,predicted,labels=range(4)).tolist())
    return metrics,target,probability

GloVe Embedding Implementation

In [ ]:
def make_pairs(corpus):
    counts = Counter()
    for sentence in corpus:
        for i, center in enumerate(sentence):
            if center == 0:
                continue
            for j in range(max(0, i - WINDOW), min(len(sentence), i + WINDOW + 1)):
                if i != j and sentence[j] != 0:
                    counts[center, sentence[j]] += 1.0 / abs(i - j)
    pairs = np.asarray(list(counts), dtype=np.int64)
    values = np.fromiter(counts.values(), dtype=np.float32, count=len(counts))
    return torch.from_numpy(pairs), torch.from_numpy(values)

def train_embedding(words, pairs, counts, epochs):
    torch.manual_seed(SEED)
    size = len(words) + 1
    # Standard GloVe two-vector representation: center word + context word
    source = nn.Embedding(size, DIMENSION, padding_idx=0, sparse=True)
    context = nn.Embedding(size, DIMENSION, padding_idx=0, sparse=True)
    nn.init.uniform_(source.weight[1:], -0.5 / DIMENSION, 0.5 / DIMENSION)
    nn.init.uniform_(context.weight[1:], -0.5 / DIMENSION, 0.5 / DIMENSION)
    # GloVe-specific bias terms
    source_bias = nn.Embedding(size, 1, sparse=True)
    context_bias = nn.Embedding(size, 1, sparse=True)
    nn.init.zeros_(source_bias.weight)
    nn.init.zeros_(context_bias.weight)
    parameters = [
        *source.parameters(), *context.parameters(),
        *source_bias.parameters(), *context_bias.parameters()
    ]
    optimizer = torch.optim.SparseAdam(parameters, lr=0.025)
    def centers(ids):
        return source(ids)
    for epoch in range(epochs):
        order = torch.randperm(len(pairs))  # Randomly shuffle all GloVe co-occurrence pairs
        loss_sum = 0.0
        for batch in order.split(4096):
            left, right = pairs[batch].T
            vectors = centers(left)
            positive = (vectors * context(right)).sum(dim=1)
            # GloVe weighted squared loss
            weights = counts[batch]
            residual = positive + source_bias(left).flatten() + context_bias(right).flatten() - weights.log()
            loss = (((weights / 10).clamp(max=1).pow(0.75)) * residual.square()).mean()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(batch)
        print(f'glove embedding epoch {epoch + 1}/{epochs} loss={loss_sum / len(order):.4f}', flush=True)
    # GloVe convention: final word vector = center-word vector + context-word vector, then normalize
    with torch.no_grad():
        vectors = torch.cat([centers(ids) for ids in torch.arange(size).split(1024)])
        vectors += context.weight
        return F.normalize(vectors, dim=1)


DPCNN Classification Model

In [ ]:
class DPCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.region = nn.Conv1d(DIMENSION,CHANNELS,KERNEL_SIZE,padding=KERNEL_SIZE//2)
        self.blocks = nn.ModuleList(nn.Sequential(nn.ReLU(),nn.Conv1d(CHANNELS,CHANNELS,KERNEL_SIZE,padding=KERNEL_SIZE//2),nn.ReLU(),nn.Conv1d(CHANNELS,CHANNELS,KERNEL_SIZE,padding=KERNEL_SIZE//2)) for _ in range(BLOCKS))
        self.output = nn.Linear(CHANNELS,4)
    def forward(self,x,mask):
        x = self.region(x.transpose(1,2))
        mask = mask[:,None,:]
        x = x.masked_fill(~mask,0)
        for i,block in enumerate(self.blocks):
            if i:
                x = F.max_pool1d(x.masked_fill(~mask,-1e9),3,2,1)
                mask = F.max_pool1d(mask.float(),3,2,1).bool()
                x = x.masked_fill(~mask,0)
            x = (x + block(x)).masked_fill(~mask,0)
        return self.output(x.masked_fill(~mask,-1e9).amax(2))

Main Workflow: GloVe Embedding + DPCNN Model Training

In [ ]:
def load_inputs(directory):
    manifest=json.loads((directory/'manifest.json').read_text(encoding='utf-8'))
    if manifest['pipeline']!='latest_clean_data_64_16_20_v2':
        raise ValueError('Use the data exported by 00_prepare_data.ipynb.')
    splits, ids = {}, {}
    for split in ('train','validation'):
        info=manifest['split_info'][split]
        path=directory/info['file']
        if hashlib.sha256(path.read_bytes()).hexdigest()!=info['sha256']:
            raise ValueError('Input hash mismatch: '+split)
        records=[json.loads(line) for line in path.read_text(encoding='utf-8').splitlines()]
        assert len(records)==info['n'] and records
        splits[split]=[]; ids[split]=[]
        for row in records:
            words=row['filtered_words']
            assert isinstance(words,list) and all(isinstance(w,str) and w for w in words)
            # Keep empty reviews in the same partition with an explicit model-input placeholder.
            splits[split].append((words or ['<empty>'],label_of(row['rating'])))
            ids[split].append(row['row_id'])
        assert set(y for _,y in splits[split])==set(range(4)), 'A partition is missing a class.'
    if any(manifest['normalized_text_group_overlap'].values()):
        print('NOTICE: supplied random split has shared normalized text groups:',manifest['normalized_text_group_overlap'])
    return splits,ids,manifest

def fit_vectors(data, manifest, cache_dir, epochs):
    # Cache signature: bind data and training configuration to ensure reproducibility
    signature = dict(
        train=manifest['split_info']['train']['sha256'],
        validation=manifest['split_info']['validation']['sha256'],
        method=EMBEDDING_METHOD,
        seed=SEED,
        dim=DIMENSION,
        window=WINDOW,
        min_count=MIN_COUNT,
        max_vocab=MAX_VOCAB,
        epochs=epochs,
        implementation='standalone_v2'
    )
    digest = hashlib.sha256(json.dumps(signature, sort_keys=True).encode()).hexdigest()
    cache_dir.mkdir(parents=True, exist_ok=True)
    path = cache_dir / (EMBEDDING_METHOD + '_' + digest + '.pt')

    # Return the cached result directly when the cache is hit
    if path.exists():
        result = torch.load(path, weights_only=True)
        assert result['signature'] == signature
        return result

    # 1. Build the vocabulary from the training set
    frequency = Counter(w for words, _ in data['train'] for w in words)
    words = [w for w, n in frequency.most_common() if n >= MIN_COUNT][:MAX_VOCAB]
    if not words:
        raise ValueError('Insufficient training vocabulary.')
    vocab = {w: i for i, w in enumerate(words, 1)}

    # 2. Build GloVe co-occurrence pairs
    pairs, counts = make_pairs([[vocab.get(w, 0) for w in row] for row, _ in data['train']])
    if pairs.numel() == 0:
        raise ValueError('No training co-occurrence pairs.')

    # 3. Train GloVe word vectors (compatible with the simplified function signature)
    vectors = train_embedding(words, pairs, counts, epochs)

    # 4. Add an unknown placeholder vector (zero vector)
    unknown = len(words) + 1
    vectors = torch.cat([vectors, torch.zeros(1, DIMENSION)])

    # 5. Assemble the result and save it to the cache
    result = dict(
        vectors=vectors,
        vocab=vocab,
        unknown=unknown,
        signature=signature
    )
    temporary = path.with_suffix('.tmp')
    torch.save(result, temporary)
    temporary.replace(path)
    return result



def main(argv=None):
    parser = argparse.ArgumentParser()
    parser.add_argument('--data-dir', type=Path, default=PACKAGE_DIR / 'data_64_16_20')
    parser.add_argument('--output-root', type=Path, default=PACKAGE_DIR / 'results')
    parser.add_argument('--epochs', type=int, default=EPOCHS)
    parser.add_argument('--embedding-epochs', type=int, default=EMBEDDING_EPOCHS)
    parser.add_argument('--weight-alpha', type=float, choices=[0.0, 0.5, 1.0], default=WEIGHT_ALPHA,
                        help='0=unweighted; 0.5=square-root; 1=inverse. MUST be identical in all six combinations.')
    args = parser.parse_args(argv)

    # Parameter validation: remove TextRCNN-specific RNN_HIDDEN validation
    assert args.epochs > 0 and args.embedding_epochs > 0
    assert KERNEL_SIZE > 0 and KERNEL_SIZE % 2 == 1 and BLOCKS > 0

    torch.set_num_threads(4)
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    # 1. Load the dataset
    data, row_ids, manifest = load_inputs(args.data_dir)

    # 2. Train or cache GloVe word vectors
    cached = fit_vectors(data, manifest, args.data_dir / 'embedding_cache', args.embedding_epochs)
    encoded = {s: encode(rows, cached['vocab'], cached['unknown']) for s, rows in data.items()}

    # 3. Initialize the DPCNN model and frozen embedding layer
    torch.manual_seed(SEED)
    model = DPCNN()
    embedding = nn.Embedding.from_pretrained(cached['vectors'], freeze=True, padding_idx=0)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    # 4. Build the data loaders
    generator = torch.Generator().manual_seed(SEED)
    training = DataLoader(encoded['train'], batch_size=BATCH_SIZE, shuffle=True, generator=generator)
    validation = DataLoader(encoded['validation'], batch_size=256)

    # 5. Calculate class-weighted loss weights
    counts = torch.bincount(encoded['train'].tensors[1], minlength=4).float()
    weights = (counts.sum() / (4 * counts)).pow(args.weight_alpha)
    weights /= weights.mean()

    # 6. Create the experiment output directory
    weight_suffix = '_sqrt' if args.weight_alpha == 0.5 else '_unweighted' if args.weight_alpha == 0 else '_inverse'
    run = args.output_root / (MODEL_NAME + '_' + EMBEDDING_METHOD + weight_suffix) / str(time.time_ns())
    run.mkdir(parents=True, exist_ok=False)

    # 7. Save the experiment configuration; remove redundant fields such as RNN_HIDDEN and BUCKETS
    config = dict(
        model=MODEL_NAME,
        embedding=EMBEDDING_METHOD,
        labels=dict(zip(NAMES, [[1, 2, 3], [4, 5, 6], [7, 8], [9, 10]])),
        input_manifest=manifest,
        seed=SEED,
        epochs=args.epochs,
        embedding_epochs=args.embedding_epochs,
        train_n=len(data['train']),
        validation_n=len(data['validation']),
        class_counts=counts.tolist(),
        embedding_dimension=DIMENSION,
        max_tokens=MAX_LENGTH,
        embedding_frozen=True,
        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        gradient_clip=GRADIENT_CLIP,
        weight_alpha=args.weight_alpha,
        weights=weights.tolist(),
        architecture=dict(
            DIMENSION=DIMENSION,
            CHANNELS=CHANNELS,
            BLOCKS=BLOCKS,
            KERNEL_SIZE=KERNEL_SIZE
        ),
        selection='best validation macro_f1',
        independent_test_evaluated=False,
        torch_version=str(torch.__version__)
    )
    (run / 'config.json').write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding='utf-8')

    # 8. Training loop
    history = []
    best = -1
    for epoch in range(1, args.epochs + 1):
        model.train()
        numerator = denominator = 0.
        start = time.perf_counter()

        for ids, y in training:
            optimizer.zero_grad()
            loss = F.cross_entropy(model(embedding(ids), ids.ne(0)), y, weight=weights)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
            optimizer.step()
            mass = weights[y].sum().item()
            numerator += loss.item() * mass
            denominator += mass

        # Validation-set evaluation
        metrics, target, probability = evaluate(model, embedding, validation)
        row = dict(epoch=epoch, train_loss=numerator / denominator, seconds=time.perf_counter() - start, **metrics)
        history.append(row)

        # Save the best model; remove redundant subword_source and used_subwords fields
        if row['macro_f1'] > best:
            best = row['macro_f1']
            torch.save(dict(
                model_state=model.state_dict(),
                vectors=cached['vectors'],
                vocab=cached['vocab'],
                unknown=cached['unknown'],
                config=config
            ), run / 'best_model.pt')
            np.savez_compressed(
                run / 'validation_predictions.npz',
                row_id=np.asarray(row_ids['validation']),
                target=target,
                probability=probability
            )

        (run / 'history.json').write_text(json.dumps(history, indent=2), encoding='utf-8')
        print(MODEL_NAME, EMBEDDING_METHOD, 'epoch', epoch,
              {k: round(row[k], 4) for k in ('accuracy', 'macro_f1', 'precision_macro', 'recall_macro')},
              flush=True)


    # 9. Verify consistency of the best-model results
    selected = max(history, key=lambda r: r['macro_f1'])
    
    checkpoint = torch.load(run / 'best_model.pt', weights_only=True)
    model.load_state_dict(checkpoint['model_state'])
    verified, _, _ = evaluate(model, embedding, validation)
    assert all(np.isclose(verified[k], selected[k], atol=1e-8, rtol=0)
               for k in ('accuracy', 'macro_f1', 'precision_macro', 'recall_macro'))

    (run / 'best_metrics.json').write_text(json.dumps(selected, indent=2), encoding='utf-8')
    print('Verified results saved to', run.resolve())
    return run



def evaluate_final_test(run_directory, data_directory):
    # Call only after all tuning is complete for final test-set evaluation
    checkpoint = torch.load(Path(run_directory) / 'best_model.pt', weights_only=True)
    cfg = checkpoint['config']

    # Verify checkpoint ownership to prevent using the wrong model
    if cfg['model'] != MODEL_NAME or cfg['embedding'] != EMBEDDING_METHOD:
        raise ValueError('Checkpoint belongs to a different combination.')

    # Restore model hyperparameters from the configuration
    globals().update(cfg['architecture'])
    globals()['MAX_LENGTH'] = cfg['max_tokens']

    # Load and verify the data manifest
    manifest = json.loads((Path(data_directory) / 'manifest.json').read_text(encoding='utf-8'))
    if manifest != cfg['input_manifest']:
        raise ValueError('Checkpoint and data manifest differ.')

    # Load the test set and verify file integrity
    info = manifest['split_info']['test']
    path = Path(data_directory) / info['file']
    if hashlib.sha256(path.read_bytes()).hexdigest() != info['sha256']:
        raise ValueError('Test hash mismatch.')

    records = [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines()]
    assert len(records) == info['n'] and records

    # Text and label processing
    rows = [(r['filtered_words'] or ['<empty>'], label_of(r['rating'])) for r in records]

    # Use the trained vocabulary and vectors directly; GloVe has no subword OOV handling, so unknown words map to unknown
    vocab = dict(checkpoint['vocab'])
    vectors = checkpoint['vectors']

    # Initialize the model and embedding layer
    model = DPCNN()
    model.load_state_dict(checkpoint['model_state'])
    embedding = nn.Embedding.from_pretrained(vectors, freeze=True, padding_idx=0)

    # Build the test-set data loader and evaluate
    loader = DataLoader(encode(rows, vocab, checkpoint['unknown']), batch_size=256)
    metrics, target, probability = evaluate(model, embedding, loader)

    # Save the results
    destination = Path(run_directory) / 'final_test_metrics.json'
    destination.write_text(json.dumps(metrics, indent=2), encoding='utf-8')
    np.savez_compressed(
        Path(run_directory) / 'final_test_predictions.npz',
        row_id=np.asarray([r['row_id'] for r in records]),
        target=target,
        probability=probability
    )

    print('FINAL TEST:', {k: metrics[k] for k in ('accuracy', 'macro_f1', 'precision_macro', 'recall_macro')})
    return metrics



In [ ]:
# Change this to the folder actually exported by the data-preparation notebook (if it is not the current directory)
DATA_DIR = PACKAGE_DIR / 'data_64_16_20'
# Pass arguments directly to main and start training
run_directory = main(['--data-dir', str(DATA_DIR)])


In [ ]:
# Keep this False during tuning. Set it to True only after all settings are finalized to evaluate the saved best model.
RUN_FINAL_TEST = True
if RUN_FINAL_TEST:
    final_test_results = evaluate_final_test(run_directory, DATA_DIR)
